In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
from tqdm import tqdm

# -------------------------
# Configuration
# -------------------------
CSV_FILE = "IMDBDataset10M.csv"

TOTAL_SAMPLES = 500  # 10000: Whole data set (Use 1000 to start)
TEST_SIZE = 0.2      # fraction for test split

MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

FINE_TUNE_BERT = False    # <<< False = frozen BERT ; True = Fine Tuning BERT

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Dataset
# -------------------------
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# Model
# -------------------------
class BertClassifier(nn.Module):
    def __init__(self, fine_tune=True):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

        if not fine_tune:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_output)

# -------------------------
# Load CSV Data
# -------------------------
def load_csv_data(path):
    df = pd.read_csv(
        path,
        header=None,
        names=["review", "label"],
        quotechar='"',
        escapechar='\\',
        engine="python"
    )

    df = df.dropna()

    df["label"] = df["label"].str.strip().str.lower()
    label_map = {"negative": 0, "positive": 1}
    df["label"] = df["label"].map(label_map)

    df = df.dropna()

    texts = df["review"].astype(str).tolist()
    labels = df["label"].astype(int).tolist()

    return texts, labels

# -------------------------
# Load and Subsample Data
# -------------------------
texts, labels = load_csv_data(CSV_FILE)
print(f"Loaded {len(texts)} samples (full dataset)")

if TOTAL_SAMPLES is not None and TOTAL_SAMPLES < len(texts):
    df_subset = pd.DataFrame({"text": texts, "label": labels}).sample(
        n=TOTAL_SAMPLES,
        random_state=42
    )
    texts = df_subset["text"].tolist()
    labels = df_subset["label"].tolist()
    print(f"Using {len(texts)} samples after subsampling")

# -------------------------
# Train / Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=labels
)

# -------------------------
# Tokenizer & Dataloaders
# -------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = TextDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset = TextDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# -------------------------
# Training Setup
# -------------------------
model = BertClassifier(fine_tune=FINE_TUNE_BERT).to(DEVICE)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

criterion = nn.CrossEntropyLoss()

# -------------------------
# Training Loop with Progress Bars
# -------------------------
def train_epoch(model, loader, epoch):
    model.train()
    total_loss = 0.0

    progress = tqdm(
        loader,
        desc=f"Epoch {epoch} [Training]",
        leave=False
    )

    for batch in progress:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)

def evaluate(model, loader, epoch):
    model.eval()
    preds, gold = [], []

    progress = tqdm(
        loader,
        desc=f"Epoch {epoch} [Testing]",
        leave=False
    )

    with torch.no_grad():
        for batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            gold.extend(labels.cpu().numpy())

    return accuracy_score(gold, preds)

# -------------------------
# Run Training
# -------------------------
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, epoch)
    test_acc = evaluate(model, test_loader, epoch)

    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"  Train loss: {train_loss:.4f}")
    print(f"  Test accuracy: {test_acc:.4f}")

print("\nFine-tuning enabled:", FINE_TUNE_BERT)
print("Total samples used:", len(texts))


Loaded 10000 samples (full dataset)
Using 500 samples after subsampling


Epoch 1/3
  Train loss: 0.7092
  Test accuracy: 0.4500


Epoch 2 [Training]:  90%|█████████ | 45/50 [10:01<00:45,  9.02s/it, loss=0.6547]